# Exercises GOLD: Text Summarization using NLP
Fill each TODO to build a graph-based extractive summarizer.

## What you'll learn
- Text cleaning (tokenize, lowercase, stopword removal).
- Word embeddings (GloVe) and sentence vectorization.
- Cosine similarity matrices and PageRank ranking.
- Extractive summarization over tennis articles.

## What you'll build
A graph-based summarizer that returns the top-ranked sentences for a set of articles.

## 0. Setup
Run installs once. If missing, download GloVe 100d from https://nlp.stanford.edu/data/glove.6B.zip and place glove.6B.100d.txt alongside the notebook.

In [ ]:
%pip install --quiet pandas numpy nltk networkx


In [ ]:
import nltk
for res in ['punkt','punkt_tab','stopwords']:
    nltk.download(res, quiet=True)


## 🌟 Exercise 1 · Data loading and inspection

In [ ]:
from pathlib import Path
data_path = 'tennis_articles.csv'  # TODO: set correct path
pdf_path = Path(data_path)
if pdf_path.suffix.lower() == '.csv':
    pdf = pd.read_csv(pdf_path, encoding='latin-1')
else:
    pdf = pd.read_excel(pdf_path)
display(pdf.head())
display(pdf.info())
if 'article_title' in pdf.columns:
    pdf = pdf.drop(columns=['article_title'])
pdf.head()


## 🌟 Exercise 2 · Sentence tokenization

In [ ]:
import nltk
sentences_list = pdf['article_text'].apply(nltk.sent_tokenize).tolist()  # TODO: adjust column name if different
sentences = [s for doc in sentences_list for s in doc]
len(sentences), sentences[:3]


## 🌟 Exercise 3 · Load GloVe embeddings

In [ ]:
%pip install --quiet kagglehub

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("danielwillgeorge/glove6b100dtxt")

print("Path to dataset files:", path)

> Impotant note: after executing previous cell, open the path folder printed and verify that the file `glove.6B.100d.txt` is indeed there. Then copy it to the same folder as this notebook for the next steps.

In [ ]:
from pathlib import Path
glove_path = Path('glove.6B.100d.txt')  # TODO: set to your local glove file
if not glove_path.exists():
    raise FileNotFoundError('Download glove.6B.100d.txt from https://nlp.stanford.edu/data/glove.6B.zip and set glove_path accordingly.')
embeddings_index = {}
with glove_path.open('r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
len(embeddings_index)


## 🌟 Exercise 4 · Text cleaning and normalization

In [ ]:
import re
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def clean_sentence(s: str) -> str:
    s = s.lower()
    s = re.sub(r'[^a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    tokens = [w for w in s.split() if w not in stop_words]
    return ' '.join(tokens)
cleaned_sentences = [clean_sentence(s) for s in sentences]
cleaned_sentences[:3]


## 🌟 Exercise 5 · Sentence vectors

In [ ]:
emb_dim = 100  # GloVe 100d
def sentence_vector(s: str):
    if not s:
        return np.zeros(emb_dim)
    words = s.split()
    vecs = [embeddings_index.get(w, np.zeros(emb_dim)) for w in words]
    return np.mean(vecs, axis=0)
sentence_vectors = np.array([sentence_vector(s) for s in cleaned_sentences])
sentence_vectors.shape


## 🌟 Exercise 6 · Similarity matrix

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
sim_mat = cosine_similarity(sentence_vectors)
sim_mat.shape


## 🌟 Exercise 7 · Graph and PageRank

In [ ]:
import networkx as nx
nx_graph = nx.from_numpy_array(sim_mat)
scores = nx.pagerank(nx_graph)
scores_list = sorted(((score, idx) for idx, score in scores.items()), reverse=True)
scores_list[:5]


## 🌟 Exercise 8 · Summarization

In [ ]:
top_n = 10  # TODO: adjust summary length
top_sentences = [sentences[idx] for _, idx in scores_list[:top_n]]
for s in top_sentences:
    print('-', s)
